# LeetCode 471 — Encode String with Shortest Length bằng Python

Độ khó trên LeetCode: Hard; theo danh mục của repo: Medium

Link đề: https://leetcode.com/problems/encode-string-with-shortest-length/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming (DP theo đoạn), Kiểm tra chu kỳ của chuỗi

Skeleton LeetCode:

```python
class Solution:
    def encode(self, s: str) -> str:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode. Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Bài có **nhiều đáp án đúng**, nên notebook kiểm tra bằng một **hàm kiểm tra hợp lệ** (giải mã lại đúng chuỗi gốc, độ dài ngắn nhất, không mã hóa thừa) và so độ dài với **oracle vét cạn** độc lập.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mã hóa chuỗi bằng luật `k[chuỗi_mã_hóa]` (lặp đúng `k` lần) sao cho kết quả **ngắn nhất**, và hiểu vì sao đáp án **không duy nhất** (bài này chấp nhận bất kỳ đáp án ngắn nhất nào).
2. Dùng các công cụ Python: nhân chuỗi `"ab" * 3`, f-string, cắt lát, `str.find`, và mẹo **kiểm tra chu kỳ** `(t + t).find(t, 1)`; viết được **bộ giải mã** bằng ngăn xếp để kiểm chứng.
3. Mô hình hóa bằng **DP theo đoạn**: `f[i][j]` = mã hóa ngắn nhất của `s[i..j]`, từ hai khả năng: **bọc cả đoạn thành `k[...]`** hoặc **cắt đôi** thành hai đoạn con tối ưu.
4. Cài đặt 4 cách khác nhau về ý tưởng: **vét cạn mọi cách mã hóa (oracle) → đệ quy có nhớ theo chuỗi con → DP đoạn bottom-up (kiểm tra chu kỳ bằng `(t+t).find`) → DP đoạn với hàm tiền tố KMP**, và so sánh độ phức tạp.
5. Tránh các lỗi kinh điển: mã hóa mà không ngắn đi, bọc chuỗi thô thay vì mã hóa tối ưu bên trong, quên bước cắt đôi, so sánh chuỗi theo thứ tự từ điển thay vì độ dài, và biết cách **tự viết hàm kiểm tra** cho bài có nhiều đáp án đúng.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Hãy **mã hóa** `s` sao cho độ dài chuỗi sau mã hóa là **ngắn nhất**.

**Luật mã hóa:** `k[encoded_string]`, trong đó `encoded_string` bên trong ngoặc vuông được lặp lại đúng `k` lần, `k` là số nguyên dương.

- Nếu việc mã hóa **không làm chuỗi ngắn đi** thì **đừng mã hóa**.
- Nếu có nhiều đáp án, trả về **bất kỳ** đáp án nào.

**Ví dụ 1:** `s = "aaa"` → `"aaa"` (không có cách mã hóa nào ngắn hơn chuỗi gốc nên giữ nguyên).

**Ví dụ 2:** `s = "aaaaa"` → `"5[a]"` (độ dài 4, ngắn hơn 5).

**Ví dụ 3:** `s = "aaaaaaaaaa"` → `"10[a]"` (độ dài 5). `"a9[a]"` hoặc `"9[a]a"` cũng đúng vì cùng độ dài 5.

**Ràng buộc:** `1 <= s.length <= 150`; `s` chỉ gồm chữ cái tiếng Anh in thường.

Vì có nhiều đáp án đúng, ta **không** so chuỗi kết quả trực tiếp với đáp án mẫu: ta kiểm tra **tính chất** của kết quả (mục 12).

In [1]:
import itertools
import random
import time
from functools import lru_cache
from typing import List

examples = [("aaa", 3), ("aaaaa", 4), ("aaaaaaaaaa", 5)]      # (chuỗi, độ dài ngắn nhất của đáp án)
for s, dai in examples:
    print(f"s = {s!r:14} -> độ dài mã hóa ngắn nhất = {dai}")

s = 'aaa'          -> độ dài mã hóa ngắn nhất = 3
s = 'aaaaa'        -> độ dài mã hóa ngắn nhất = 4
s = 'aaaaaaaaaa'   -> độ dài mã hóa ngắn nhất = 5


## 3. Kiến thức nền

### 3.1. Nhân chuỗi, f-string và độ dài

`"ab" * 3` ra `"ababab"` (lặp chuỗi 3 lần). f-string `f"{k}[{e}]"` ghép số `k` và chuỗi `e` thành `"k[e]"`. Độ dài của `"k[e]"` là `len(str(k)) + 2 + len(e)`: số chữ số của `k`, hai dấu ngoặc, và chuỗi `e`.

In [2]:
print("ab" * 3)                                   # ababab
k, e = 10, "a"
ma = f"{k}[{e}]"
print(ma, len(ma), len(str(k)) + 2 + len(e))       # 10[a] 5 5
print(len("aaaaaaaaaa"), "->", len(ma))            # 10 -> 5 : ngắn hơn

ababab
10[a] 5 5
10 -> 5


### 3.2. Giải mã bằng ngăn xếp (stack)

Để biết một chuỗi mã hóa có đúng không, ta cần **giải mã** nó: `"2[ab]c"` → `"ababc"`. Quy tắc: đọc từ trái sang phải; gặp chữ số thì gom thành số `k`; gặp `[` thì **cất** phần đang dựng và `k` vào ngăn xếp rồi bắt đầu phần mới; gặp `]` thì **lấy** phần cũ ra và nối phần trong ngoặc lặp `k` lần. Ngăn xếp (stack) là danh sách mà ta chỉ thêm/bỏ ở cuối (`append`/`pop`).

In [3]:
def giai_ma_don_gian(e):
    stack = []              # mỗi phần tử: (phần đã dựng trước dấu '[', số k)
    cur = ""
    num = 0
    for ch in e:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, num))
            cur, num = "", 0
        elif ch == "]":
            truoc, k = stack.pop()
            cur = truoc + cur * k
        else:
            cur += ch
    return cur

print(giai_ma_don_gian("2[ab]c"))                  # ababc
print(giai_ma_don_gian("3[a2[c]]"))                # accaccacc
print(giai_ma_don_gian("2[2[abbb]c]"))             # abbbabbbcabbbabbbc

ababc
accaccacc
abbbabbbcabbbabbbc


### 3.3. Chu kỳ của chuỗi và mẹo `(t + t).find(t, 1)`

Chuỗi `t` có **chu kỳ** `p` (với `p` là ước của `len(t)`, `p < len(t)`) nếu `t = u * (len(t)//p)` với `u = t[:p]`. Ví dụ `"abcabc"` có chu kỳ `3`.

**Mẹo:** tìm vị trí **sớm nhất `>= 1`** mà `t` xuất hiện trong `t + t`. Nếu kết quả `p < len(t)` thì `p` chính là **chu kỳ nhỏ nhất** của `t` (và `len(t) % p == 0`); nếu `p == len(t)` thì `t` không lặp. Lý do: dịch vòng `t` đi `p` vị trí mà vẫn ra `t` ⇔ `t` lặp với chu kỳ `p`.

In [4]:
for t in ["abcabc", "aaaaa", "abab", "abcab", "abcd", "aabaab"]:
    p = (t + t).find(t, 1)
    print(f"{t!r:10} -> p = {p}  {'lặp ' + str(len(t) // p) + ' lần ' + repr(t[:p]) if p < len(t) else 'không lặp'}")

'abcabc'   -> p = 3  lặp 2 lần 'abc'
'aaaaa'    -> p = 1  lặp 5 lần 'a'
'abab'     -> p = 2  lặp 2 lần 'ab'
'abcab'    -> p = 5  không lặp
'abcd'     -> p = 4  không lặp
'aabaab'   -> p = 3  lặp 2 lần 'aab'


### 3.4. Bảng 2 chiều theo đoạn `[i, j]` và cắt lát `s[i:j+1]`

`f[i][j]` sẽ lưu mã hóa ngắn nhất của đoạn `s[i..j]` (cả hai đầu tính). Đoạn đó là `s[i:j+1]`. Muốn tính `f[i][j]` ta cần các đoạn **nhỏ hơn nằm trong nó**, nên duyệt `i` từ **phải sang trái** (`n-1` về `0`) và `j` từ `i` tăng dần.

In [5]:
s = "abcde"
n = len(s)
for i in range(n - 1, -1, -1):
    print(f"i = {i}:", [s[i:j + 1] for j in range(i, n)])

i = 4: ['e']
i = 3: ['d', 'de']
i = 2: ['c', 'cd', 'cde']
i = 1: ['b', 'bc', 'bcd', 'bcde']
i = 0: ['a', 'ab', 'abc', 'abcd', 'abcde']


### 3.5. Ghi nhớ kết quả (memo) bằng `dict`

Khi cùng một chuỗi con bị xử lý nhiều lần, ta lưu kết quả vào `dict` (khóa là **chuỗi con**) và tra cứu thay vì tính lại. Ở Cách 2 ta dùng khóa là chuỗi; ở Cách 3/4 dùng bảng chỉ số `(i, j)`.

In [6]:
memo = {}
def dem_lan_tinh(t):
    if t in memo:
        return memo[t]
    memo[t] = len(t)          # ví dụ "tính toán đắt tiền"
    return memo[t]

dem_lan_tinh("abc"); dem_lan_tinh("abc"); dem_lan_tinh("de")
print(memo)                    # {'abc': 3, 'de': 2}

{'abc': 3, 'de': 2}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Chuỗi nào mới đáng mã hóa?** `k[e]` tốn ít nhất `4` ký tự (`k` một chữ số, hai ngoặc, `e` một ký tự). Vì vậy chuỗi **dài `<= 4`** không bao giờ ngắn đi: giữ nguyên. Chuỗi dài `5` trở lên mới cần xét.

**Ví dụ `s = "aaaaa"` (5 chữ `a`):** lặp `5` lần `"a"` → `"5[a]"` dài `4 < 5` ✔. Đáp án `"5[a]"`.

**Ví dụ `s = "aaaaaaaaaa"` (10 chữ `a`):** `"10[a]"` dài `5`. Cũng có thể `"a" + "9[a]"` = `"a9[a]"` dài `5` (cắt đôi: `1` chữ rồi `9` chữ). Hai cách ngang nhau ⇒ **nhiều đáp án đúng**.

**Ví dụ lồng nhau `s = "abbbabbbcabbbabbbc"` (18 ký tự):** toàn chuỗi là `"abbbabbbc"` lặp 2 lần; bên trong `"abbbabbbc"` là `"abbb"` lặp 2 lần rồi `"c"`. Mã hóa tối ưu `"2[2[abbb]c]"` dài `11`. Nếu chỉ bọc chuỗi thô: `"2[abbbabbbc]"` dài `12` ✘ (không tối ưu). Nghĩa là **bên trong ngoặc cũng phải là mã hóa tối ưu** của phần được lặp.

**Hai kiểu "lựa chọn" cho một đoạn `t`:**

1. **Bọc cả đoạn:** nếu `t = u` lặp `k` lần (`k >= 2`) thì ứng viên `k[mã hóa tối ưu của u]`.
2. **Cắt đôi:** chọn một điểm cắt, ghép `mã hóa tối ưu(trái) + mã hóa tối ưu(phải)`.

Và luôn có thể **giữ nguyên** `t`. Lấy ứng viên **ngắn nhất**.

In [7]:
print(len("abbbabbbcabbbabbbc"), len("2[abbbabbbc]"), len("2[2[abbb]c]"))     # 18 12 11
print(giai_ma_don_gian("2[2[abbb]c]") == "abbbabbbcabbbabbbc")                 # True
print(giai_ma_don_gian("a9[a]") == giai_ma_don_gian("10[a]") == "a" * 10)      # True: nhiều đáp án đúng

18 12 11
True
True


## 5. Cách 1 — Brute Force / baseline: sinh MỌI cách mã hóa rồi chọn cái ngắn nhất

**Ý tưởng:** dựa **thẳng vào định nghĩa**. Một chuỗi `t` có thể mã hóa theo 3 kiểu:

1. giữ nguyên `t`;
2. nếu `t = u` lặp `k >= 2` lần thì `k[e]` với **mọi** cách mã hóa `e` của `u`;
3. cắt `t` thành hai phần không rỗng, mã hóa **mọi cách** từng phần rồi ghép.

Ta **sinh tập tất cả** chuỗi mã hóa hợp lệ của `s`, lọc các cái **không có khối nào mã hóa thừa** (mục 12 định nghĩa hàm kiểm tra) rồi chọn cái ngắn nhất. Chỉ dùng được với chuỗi rất ngắn, nhưng **độc lập hoàn toàn** với DP nên rất tốt làm **oracle**.

In [8]:
@lru_cache(maxsize=None)
def sinh_tat_ca(t):
    """Tập mọi chuỗi mã hóa của t (kể cả những cái không làm ngắn đi)."""
    n = len(t)
    kq = {t}
    for k in range(2, n + 1):                          # t = u * k
        if n % k == 0 and t[: n // k] * k == t:
            for e in sinh_tat_ca(t[: n // k]):
                kq.add(f"{k}[{e}]")
    for i in range(1, n):                              # cắt đôi
        for a in sinh_tat_ca(t[:i]):
            for b in sinh_tat_ca(t[i:]):
                kq.add(a + b)
    return frozenset(kq)

print(len(sinh_tat_ca("aaaaaa")), sorted(sinh_tat_ca("aaaaaa"), key=lambda x: (len(x), x))[:6])
print(sorted(sinh_tat_ca("abab"), key=lambda x: (len(x), x)))

48 ['6[a]', '3[aa]', '5[a]a', 'a5[a]', '2[aaa]', '4[a]aa']
['abab', '2[ab]']


Để chọn "ngắn nhất và hợp lệ" ta cần hàm kiểm tra. Ta dựng **bộ phân tích/giải mã chặt chẽ** (ngăn xếp), trả về chuỗi giải mã và số **khối mã hóa thừa** (khối mà `len(khối) >= len(nội dung sau khi giải mã)`), hoặc `None` nếu sai cú pháp.

In [9]:
def phan_tich(e):
    """Phân tích chuỗi mã hóa e. Trả về (chuỗi_giải_mã, số_khối_không_làm_ngắn) hoặc None nếu sai cú pháp."""
    stack = []
    cur = []
    num = ""
    start = 0
    vi_pham = 0
    for p, ch in enumerate(e):
        if ch.isdigit() and ch.isascii():
            if num == "":
                if ch == "0":
                    return None                        # k phải là số nguyên dương, không bắt đầu bằng 0
                start = p
            num += ch
        elif ch == "[":
            if num == "":
                return None                            # '[' phải đứng sau số k
            stack.append((cur, int(num), start))
            cur, num = [], ""
        elif ch == "]":
            if num or not stack or not cur:
                return None                            # ']' sai chỗ hoặc ngoặc rỗng
            truoc, k, st = stack.pop()
            giai = "".join(cur) * k
            if p - st + 1 >= len(giai):                # độ dài khối k[...] không nhỏ hơn bản giải mã
                vi_pham += 1
            truoc.append(giai)
            cur = truoc
        elif "a" <= ch <= "z":
            if num:
                return None                            # chữ cái ngay sau số (thiếu '[')
            cur.append(ch)
        else:
            return None
    if stack or num:
        return None
    return "".join(cur), vi_pham

print(phan_tich("2[2[abbb]c]"))       # ('abbbabbbcabbbabbbc', 0)
print(phan_tich("4[a]"))              # ('aaaa', 1)   <- khối thừa: độ dài 4 không nhỏ hơn 4
print(phan_tich("2[ab"), phan_tich("2ab]"), phan_tich("0[a]"), phan_tich("[a]"))     # None x4

('abbbabbbcabbbabbbc', 0)
('aaaa', 1)
None None None None


In [10]:
def nghiem_ngat_ok(e):
    pt = phan_tich(e)
    return pt is not None and pt[1] == 0

def encode_vet_can(s):
    """Cách 1: sinh mọi cách mã hóa, lọc 'không mã hóa thừa', chọn ngắn nhất."""
    ung_vien = [e for e in sinh_tat_ca(s) if nghiem_ngat_ok(e)]
    return min(ung_vien, key=lambda e: (len(e), e))

for s, dai in examples:
    kq = encode_vet_can(s)
    print(s, "->", kq, len(kq))
    assert len(kq) == dai and giai_ma_don_gian(kq) == s

aaa -> aaa 3
aaaaa -> 5[a] 4
aaaaaaaaaa -> 10[a] 5


**Độ phức tạp:** số cách mã hóa tăng **rất nhanh** theo độ dài (tích các lựa chọn ở mọi điểm cắt), cỡ hàm mũ. Chỉ chạy được tới độ dài khoảng `10`–`12`.

**Nút thắt:** hãy đếm số cách mã hóa khi chuỗi dài ra.

In [11]:
for n in (4, 6, 8, 10, 12):
    print(f"n = {n:>2}: {len(sinh_tat_ca('a' * n)):>7} cách mã hóa của {'a' * n!r:16}")

n =  4:      10 cách mã hóa của 'aaaa'          
n =  6:      48 cách mã hóa của 'aaaaaa'        
n =  8:     230 cách mã hóa của 'aaaaaaaa'      
n = 10:    1118 cách mã hóa của 'aaaaaaaaaa'    
n = 12:    5450 cách mã hóa của 'aaaaaaaaaaaa'  


Số cách tăng cực nhanh, và với `n = 150` là không thể. Nhưng nhận xét: ta chỉ cần **mã hóa ngắn nhất cho mỗi chuỗi con**, vì ghép các phần tối ưu cho kết quả tối ưu. Có `O(n²)` chuỗi con khác nhau. Đó là bước đột phá.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Cấu trúc con tối ưu.** Gọi `f(t)` là mã hóa ngắn nhất của `t`. Một mã hóa của `t` là (a) chính `t`, hoặc (b) một khối duy nhất `k[e]` bọc cả `t`, hoặc (c) ghép hai phần của mã hóa cắt `t` làm hai. Trong (b), `e` nên là mã hóa **ngắn nhất** của `u`; trong (c), mỗi phần nên là mã hóa ngắn nhất của phần đó. Vậy `f(t)` chỉ phụ thuộc `f` của các chuỗi con **ngắn hơn**.
2. **Chỉ `O(n²)` đoạn khác nhau** `[i, j]` ⇒ DP theo đoạn `f[i][j]`.
3. **Khối bọc cả đoạn:** nếu `t = u` lặp `k` lần, dùng `u` ngắn nhất (chu kỳ nhỏ nhất) ⇒ ứng viên `k[f(u)]` với `u = s[i : i + p]`, và đoạn `u` đã được tính (nằm bên trong, ngắn hơn). Kiểm tra chu kỳ bằng `(t + t).find(t, 1)` hoặc bằng hàm tiền tố KMP.
4. **Cắt đôi:** thử mọi điểm cắt `m ∈ [i, j-1]`: `f[i][m] + f[m+1][j]`.
5. **Chọn ngắn nhất theo độ dài**, chỉ thay khi **ngắn đi thật sự** (đề: "không làm ngắn đi thì đừng mã hóa"). Đoạn dài `< 5` giữ nguyên.
6. **Thứ tự tính:** `i` từ phải sang trái và `j` từ trái sang phải để mọi đoạn con đều sẵn sàng.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ theo chuỗi con (top-down), thử mọi ước làm chu kỳ

**Giải thích từng đoạn:**

- `enc(t)` trả mã hóa ngắn nhất của chuỗi `t`; kết quả lưu trong `memo[t]` (khóa là chính chuỗi con).
- `best = t`; nếu `len(t) < 5` trả luôn `t`.
- **Bọc cả chuỗi:** thử **mọi ước `d`** của `len(t)` với `d < len(t)`: nếu `t[:d] * (len(t)//d) == t` thì ứng viên `f"{len(t)//d}[{enc(t[:d])}]"`. (Ta không giả định chỉ chu kỳ nhỏ nhất mới tốt: thử mọi ước, rồi kiểm chứng ở Cách 3/4.)
- **Cắt đôi:** với mọi `i`, so **độ dài** `len(enc(a)) + len(enc(b))` với `len(best)` trước, chỉ ghép chuỗi khi ngắn hơn.

In [12]:
def encode_memo(s):
    memo = {}

    def enc(t):
        if t in memo:
            return memo[t]
        n = len(t)
        best = t
        if n >= 5:
            for d in range(1, n):                              # mọi ước d làm chu kỳ
                if n % d == 0 and t[:d] * (n // d) == t:
                    cand = f"{n // d}[{enc(t[:d])}]"
                    if len(cand) < len(best):
                        best = cand
            for i in range(1, n):                              # cắt đôi
                a, b = enc(t[:i]), enc(t[i:])
                if len(a) + len(b) < len(best):
                    best = a + b
        memo[t] = best
        return best

    return enc(s)

for s, dai in examples:
    kq = encode_memo(s)
    print(s, "->", kq, len(kq))
    assert len(kq) == dai and giai_ma_don_gian(kq) == s
print(encode_memo("abbbabbbcabbbabbbc"))

aaa -> aaa 3
aaaaa -> 5[a] 4
aaaaaaaaaa -> 10[a] 5
2[2[abbb]c]


In [13]:
class SolutionMemo:
    def encode(self, s: str) -> str:
        memo = {}

        def enc(t: str) -> str:
            if t in memo:
                return memo[t]
            n = len(t)
            best = t
            if n >= 5:
                for d in range(1, n):
                    if n % d == 0 and t[:d] * (n // d) == t:
                        cand = f"{n // d}[{enc(t[:d])}]"
                        if len(cand) < len(best):
                            best = cand
                for i in range(1, n):
                    a, b = enc(t[:i]), enc(t[i:])
                    if len(a) + len(b) < len(best):
                        best = a + b
            memo[t] = best
            return best

        return enc(s)

print(SolutionMemo().encode("aaaaaaaaaa"))

10[a]


**Độ phức tạp (Cách 2):** có `O(n²)` chuỗi con khác nhau; mỗi chuỗi tốn `O(n)` điểm cắt (so độ dài `O(1)`, ghép chuỗi hiếm khi) và `O(n)` ước × `O(n)` so sánh chuỗi cho phần chu kỳ ⇒ khoảng `O(n³)` thời gian (tính theo số bước cơ bản) với `n ≤ 150` là chấp nhận được; bộ nhớ `O(n²)` chuỗi con (các chuỗi mã hóa ngắn).

### Cách 3 — DP đoạn bottom-up, kiểm tra chu kỳ bằng `(t + t).find(t, 1)` (cách chuẩn)

**Giải thích từng đoạn:**

- `f = [[""] * n for _ in range(n)]`; duyệt `i` từ `n-1` về `0`, `j` từ `i` đến `n-1`; `t = s[i:j+1]`, `L = j - i + 1`.
- Khởi tạo `f[i][j] = t`.
- Nếu `L >= 5`:
  - **Bọc cả đoạn:** `p = (t + t).find(t, 1)`; nếu `p < L` thì `cand = f"{L//p}[{f[i][i+p-1]}]"` (đoạn `[i, i+p-1]` ngắn hơn nên đã tính). Nếu `len(cand) < len(f[i][j])` thì lấy `cand`.
  - **Cắt đôi:** với `m` từ `i` đến `j-1`: nếu `len(f[i][m]) + len(f[m+1][j]) < len(f[i][j])` thì `f[i][j] = f[i][m] + f[m+1][j]`.
- Đáp án `f[0][n-1]`.

In [14]:
def encode_dp(s):
    n = len(s)
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            L = j - i + 1
            f[i][j] = t
            if L >= 5:
                p = (t + t).find(t, 1)                         # chu kỳ nhỏ nhất (nếu p < L)
                if p < L:
                    cand = f"{L // p}[{f[i][i + p - 1]}]"
                    if len(cand) < len(f[i][j]):
                        f[i][j] = cand
                for m in range(i, j):
                    if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                        f[i][j] = f[i][m] + f[m + 1][j]
    return f[0][n - 1]

for s, dai in examples:
    kq = encode_dp(s)
    print(s, "->", kq, len(kq))
    assert len(kq) == dai and giai_ma_don_gian(kq) == s
print(encode_dp("abbbabbbcabbbabbbc"), encode_dp("aabcaabcd"))

aaa -> aaa 3
aaaaa -> 5[a] 4
aaaaaaaaaa -> 10[a] 5
2[2[abbb]c] 2[aabc]d


In [15]:
class SolutionDP:
    def encode(self, s: str) -> str:
        n = len(s)
        f = [[""] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            for j in range(i, n):
                t = s[i : j + 1]
                f[i][j] = t
                if j - i + 1 >= 5:
                    p = (t + t).find(t, 1)
                    if p < len(t):
                        cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
                        if len(cand) < len(f[i][j]):
                            f[i][j] = cand
                    for m in range(i, j):
                        if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                            f[i][j] = f[i][m] + f[m + 1][j]
        return f[0][n - 1]

print(SolutionDP().encode("aaaaa"))

5[a]


**Độ phức tạp (Cách 3):** `O(n²)` đoạn; mỗi đoạn tốn `O(L)` cho `(t+t).find` (thực tế nhanh vì hàm tìm của Python viết bằng C) và `O(L)` điểm cắt ⇒ `O(n³)` thời gian; bộ nhớ `O(n²)` ô lưu chuỗi mã hóa (mỗi chuỗi dài tối đa `L`, thực tế rất ngắn vì là mã hóa).

### Cách 4 — DP đoạn với hàm tiền tố KMP (tìm chu kỳ trong `O(1)` mỗi đoạn)

**Ý tưởng khác:** thay vì gọi `find` trên chuỗi con mỗi đoạn, với mỗi điểm đầu `i` ta **duy trì mảng tiền tố (prefix function) `pi` khi `j` tăng dần**. `pi[L-1]` là độ dài **tiền tố dài nhất đồng thời là hậu tố** (không tính cả chuỗi) của `s[i : i+L]`. Khi đó **chu kỳ nhỏ nhất** là `p = L - pi[L-1]`; chuỗi lặp hoàn toàn (`k = L // p >= 2`) ⇔ `p < L` và `L % p == 0`.

Mỗi đoạn kiểm tra chu kỳ chỉ tốn `O(1)` (khấu hao): tổng việc kiểm tra chu kỳ toàn bài là `O(n²)`.

**Giải thích từng đoạn:** với mỗi `i`, tạo `pi = [0] * (n - i)`. Khi thêm ký tự `s[j]` (độ dài mới `L = j - i + 1`), cập nhật `pi[L-1]` theo cách chuẩn của KMP: `k = pi[L-2]`, lùi `k = pi[k-1]` khi `s[i+k] != s[j]`, nếu khớp thì `k += 1`.

In [16]:
def encode_kmp(s):
    n = len(s)
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        pi = [0] * (n - i)                                     # hàm tiền tố của s[i:]
        for j in range(i, n):
            L = j - i + 1
            if L > 1:
                k = pi[L - 2]
                while k > 0 and s[i + k] != s[j]:
                    k = pi[k - 1]
                if s[i + k] == s[j]:
                    k += 1
                pi[L - 1] = k
            f[i][j] = s[i:j + 1]
            if L >= 5:
                p = L - pi[L - 1]                              # chu kỳ nhỏ nhất
                if p < L and L % p == 0:
                    cand = f"{L // p}[{f[i][i + p - 1]}]"
                    if len(cand) < len(f[i][j]):
                        f[i][j] = cand
                for m in range(i, j):
                    if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                        f[i][j] = f[i][m] + f[m + 1][j]
    return f[0][n - 1]

for s, dai in examples:
    kq = encode_kmp(s)
    print(s, "->", kq, len(kq))
    assert len(kq) == dai and giai_ma_don_gian(kq) == s
print(encode_kmp("abbbabbbcabbbabbbc"), encode_kmp("aabcaabcd"))

aaa -> aaa 3
aaaaa -> 5[a] 4
aaaaaaaaaa -> 10[a] 5
2[2[abbb]c] 2[aabc]d


In [17]:
class SolutionKMP:
    def encode(self, s: str) -> str:
        n = len(s)
        f = [[""] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            pi = [0] * (n - i)
            for j in range(i, n):
                L = j - i + 1
                if L > 1:
                    k = pi[L - 2]
                    while k > 0 and s[i + k] != s[j]:
                        k = pi[k - 1]
                    if s[i + k] == s[j]:
                        k += 1
                    pi[L - 1] = k
                f[i][j] = s[i : j + 1]
                if L >= 5:
                    p = L - pi[L - 1]
                    if p < L and L % p == 0:
                        cand = f"{L // p}[{f[i][i + p - 1]}]"
                        if len(cand) < len(f[i][j]):
                            f[i][j] = cand
                    for m in range(i, j):
                        if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                            f[i][j] = f[i][m] + f[m + 1][j]
        return f[0][n - 1]

print(SolutionKMP().encode("aaaaa"))

5[a]


**Độ phức tạp (Cách 4):** kiểm tra chu kỳ toàn cục `O(n²)`, nhưng bước **cắt đôi** vẫn `O(n)` mỗi đoạn ⇒ tổng vẫn `O(n³)` thời gian (hằng số nhỏ hơn đôi chút); bộ nhớ `O(n²)`. Ưu điểm: hiểu rõ "chu kỳ nhỏ nhất = `L - pi[L-1]`" — công cụ hay cho nhiều bài chuỗi (xem Biến thể A).

## 8. TRACE từng bước

Trace **Cách 3** với `s = "aabcaabcd"` (9 ký tự). Chỉ các đoạn dài `>= 5` mới có thể mã hóa. Ta in cho từng đoạn: chu kỳ nhỏ nhất, ứng viên bọc, các lần cắt đôi cải thiện.

Các đoạn dài `>= 5` quan trọng:

| đoạn `[i,j]` | `t` | chu kỳ `p` | ứng viên bọc | cắt đôi cải thiện? | `f[i][j]` |
|---|---|---|---|---|---|
| [0,7] | `aabcaabc` | 4 | `2[aabc]` (7 < 8) ✔ | không ngắn hơn 7 | `2[aabc]` |
| [0,8] | `aabcaabcd` | 9 (không lặp) | — | cắt sau vị trí 7: `2[aabc]` + `d` = 8 < 9 ✔ | `2[aabc]d` |

Hàm `trace_dp` bên dưới in chi tiết tất cả các đoạn có độ dài `>= 5`.

In [18]:
def trace_dp(s):
    n = len(s)
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            L = j - i + 1
            f[i][j] = t
            if L < 5:
                continue
            p = (t + t).find(t, 1)
            nhat = [f"đoạn [{i},{j}] t={t!r} (L={L}): chu kỳ nhỏ nhất p={p}"]
            if p < L:
                cand = f"{L // p}[{f[i][i + p - 1]}]"
                nhat.append(f"bọc: {cand!r} (dài {len(cand)} vs {len(f[i][j])})")
                if len(cand) < len(f[i][j]):
                    f[i][j] = cand
                    nhat.append("-> nhận ứng viên bọc")
            else:
                nhat.append("không lặp: không bọc được")
            for m in range(i, j):
                a, b = f[i][m], f[m + 1][j]
                if len(a) + len(b) < len(f[i][j]):
                    f[i][j] = a + b
                    nhat.append(f"cắt sau {m}: {a!r} + {b!r} = {f[i][j]!r} (dài {len(f[i][j])}) -> cải thiện")
            nhat.append(f"=> f[{i}][{j}] = {f[i][j]!r}")
            print("\n   ".join(nhat))
    return f[0][n - 1]

print("KẾT QUẢ:", trace_dp("aabcaabcd"))

đoạn [4,8] t='aabcd' (L=5): chu kỳ nhỏ nhất p=5
   không lặp: không bọc được
   => f[4][8] = 'aabcd'
đoạn [3,7] t='caabc' (L=5): chu kỳ nhỏ nhất p=5
   không lặp: không bọc được
   => f[3][7] = 'caabc'
đoạn [3,8] t='caabcd' (L=6): chu kỳ nhỏ nhất p=6
   không lặp: không bọc được
   => f[3][8] = 'caabcd'
đoạn [2,6] t='bcaab' (L=5): chu kỳ nhỏ nhất p=5
   không lặp: không bọc được
   => f[2][6] = 'bcaab'
đoạn [2,7] t='bcaabc' (L=6): chu kỳ nhỏ nhất p=6
   không lặp: không bọc được
   => f[2][7] = 'bcaabc'
đoạn [2,8] t='bcaabcd' (L=7): chu kỳ nhỏ nhất p=7
   không lặp: không bọc được
   => f[2][8] = 'bcaabcd'
đoạn [1,5] t='abcaa' (L=5): chu kỳ nhỏ nhất p=5
   không lặp: không bọc được
   => f[1][5] = 'abcaa'
đoạn [1,6] t='abcaab' (L=6): chu kỳ nhỏ nhất p=6
   không lặp: không bọc được
   => f[1][6] = 'abcaab'
đoạn [1,7] t='abcaabc' (L=7): chu kỳ nhỏ nhất p=7
   không lặp: không bọc được
   => f[1][7] = 'abcaabc'
đoạn [1,8] t='abcaabcd' (L=8): chu kỳ nhỏ nhất p=8
   không lặp: không bọc đư

In [19]:
# Trace "aaaaaaaaaa" (10 chữ a): chỉ in kết quả các đoạn bắt đầu tại 0 để thấy 10[a] (bọc) cạnh tranh với cắt đôi
def trace_a(n):
    s = "a" * n
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            f[i][j] = t
            if len(t) >= 5:
                p = (t + t).find(t, 1)
                if p < len(t):
                    cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
                    if len(cand) < len(f[i][j]):
                        f[i][j] = cand
                for m in range(i, j):
                    if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                        f[i][j] = f[i][m] + f[m + 1][j]
    for j in range(n):
        print(f"f[0][{j}] = {f[0][j]!r:10} dài {len(f[0][j])}")

trace_a(10)

f[0][0] = 'a'        dài 1
f[0][1] = 'aa'       dài 2
f[0][2] = 'aaa'      dài 3
f[0][3] = 'aaaa'     dài 4
f[0][4] = '5[a]'     dài 4
f[0][5] = '6[a]'     dài 4
f[0][6] = '7[a]'     dài 4
f[0][7] = '8[a]'     dài 4
f[0][8] = '9[a]'     dài 4
f[0][9] = '10[a]'    dài 5


## 9. Code nộp LeetCode

Dùng Cách 3 (DP đoạn bottom-up với `(t + t).find(t, 1)`): ngắn, nhanh với `n <= 150`.

In [20]:
class Solution:
    def encode(self, s: str) -> str:
        n = len(s)
        # f[i][j]: mã hóa ngắn nhất của s[i..j]
        f = [[""] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            for j in range(i, n):
                t = s[i : j + 1]
                f[i][j] = t                                    # mặc định: giữ nguyên
                if j - i + 1 < 5:
                    continue                                   # chuỗi ngắn: mã hóa không bao giờ ngắn hơn
                p = (t + t).find(t, 1)                         # chu kỳ nhỏ nhất của t
                if p < len(t):                                 # t = (t[:p]) lặp len(t)//p lần
                    cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
                    if len(cand) < len(f[i][j]):
                        f[i][j] = cand
                for m in range(i, j):                          # cắt đôi
                    if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                        f[i][j] = f[i][m] + f[m + 1][j]
        return f[0][n - 1]


for s, dai in examples:
    kq = Solution().encode(s)
    print(s, "->", kq)
    assert len(kq) == dai and giai_ma_don_gian(kq) == s

aaa -> aaa
aaaaa -> 5[a]
aaaaaaaaaa -> 10[a]


## 10. Vì sao thuật toán đúng

**Khung mã hóa.** Một chuỗi mã hóa hợp lệ của `t` là **ghép (concatenation)** của các **mục**; mỗi mục là một chữ cái hoặc một khối `k[e]`. Vì vậy với `t` bất kỳ, mã hóa `E` của nó thuộc đúng một trong ba loại: (a) `E = t` (toàn chữ cái, không khối nào); (b) `E` là **một khối duy nhất** `k[e]` bao cả `t` (khi đó `t = u` lặp `k` lần và `e` mã hóa `u`); (c) `E` gồm **từ 2 mục trở lên** và có thể cắt thành hai phần `E = E1 + E2` (tại ranh giới giữa hai mục) tương ứng với cách cắt `t = t1 + t2`.

**Cấu trúc con tối ưu.** Trong (b), đổi `e` bằng mã hóa ngắn nhất của `u` không làm dài thêm. Trong (c), đổi `E1`, `E2` bằng mã hóa ngắn nhất của `t1`, `t2` không làm dài thêm. Vậy `f(t)` = ngắn nhất trong ba loại ứng viên, với phần bên trong lấy từ `f` của chuỗi con **ngắn hơn**. Đó là đúng công thức DP của ta. Khi DP duyệt mọi cách cắt, ứng viên (c) được phủ hết; (a) là giá trị khởi tạo; (b) là bước "bọc".

**Chu kỳ nhỏ nhất là đủ cho (b).** Dùng chu kỳ nhỏ nhất `p` (thay vì mọi ước) là đủ: nếu `u = p^m` thì khối `k[f(u)]` không ngắn hơn ứng viên `(k·m)[f(p)]` hoặc một cách cắt khác mà DP đã xét (số chữ số của `k·m` không vượt tổng số chữ số của `k` và `m`). Ta **không dựa vào lập luận đó mà kiểm chứng bằng đối chiếu**: Cách 2 thử **mọi ước** làm chu kỳ còn Cách 3/4 chỉ chu kỳ nhỏ nhất, và độ dài kết quả luôn bằng nhau (mục 12), đồng thời bằng oracle vét cạn trên mọi chuỗi ngắn.

**Điều kiện "ngắn đi thật sự".** Ta chỉ thay `f[i][j]` khi ứng viên **ngắn hơn** (`<` chứ không `<=`). Nhờ vậy mọi khối `k[e]` trong đáp án đều **ngắn hơn** bản giải mã của nó, đúng yêu cầu "không ngắn đi thì đừng mã hóa" (hàm kiểm tra `nghiem_ngat_ok` ở mục 5 kiểm đúng điều này). Chuỗi ngắn `< 5` không bao giờ có khối ngắn hơn bản giải mã: mọi khối `k[e]` dài ít nhất `4` ký tự, nên để *ngắn hơn* bản giải mã thì bản giải mã phải dài ít nhất `5` (ví dụ `5[a]` ↔ `aaaaa`). ∎

In [21]:
# Kiểm chứng nhận xét "chuỗi dài <= 4 không bao giờ mã hóa ngắn hơn": mọi khối hợp lệ bắt buộc có độ dài >= 5 sau giải mã
min_giai_ma = min(len(phan_tich(e)[0]) for e in ["5[a]", "3[ab]", "2[abc]", "6[a]"] if phan_tich(e)[1] == 0)
print("bản giải mã ngắn nhất của một khối 'thật sự ngắn hơn' trong các ví dụ:", min_giai_ma)
for t in ["aaaa", "abab", "aaa", "aa", "a", "abcd"]:
    assert all(not nghiem_ngat_ok(e) or e == t for e in sinh_tat_ca(t)), t
print("với mọi chuỗi dài <= 4 thử nghiệm: không có cách mã hóa nào hợp lệ khác chính nó")

bản giải mã ngắn nhất của một khối 'thật sự ngắn hơn' trong các ví dụ: 5
với mọi chuỗi dài <= 4 thử nghiệm: không có cách mã hóa nào hợp lệ khác chính nó


## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Sinh mọi cách mã hóa | hàm mũ | hàm mũ |
| 2. Đệ quy nhớ theo chuỗi con (mọi ước) | `~O(n³)`–`O(n⁴)` bước (ký tự), nhanh trong thực tế | `O(n²)` chuỗi |
| 3. DP đoạn + `(t+t).find` | `O(n³)` | `O(n²)` |
| 4. DP đoạn + KMP | `O(n³)` (kiểm tra chu kỳ `O(n²)`) | `O(n²)` |

Với `n = 150`: số đoạn `~ 11 000`, mỗi đoạn tối đa `150` điểm cắt ⇒ cỡ `1.7 × 10⁶` phép so sánh độ dài: chạy trong vài trăm mili-giây.

(Về bộ nhớ: mỗi ô `f[i][j]` là chuỗi có độ dài tối đa bằng `j - i + 1` nhưng thường rất ngắn nhờ mã hóa; tổng tối đa `O(n³)` ký tự trong trường hợp xấu nhất, `n = 150` nhỏ nên không đáng lo.)

## 12. Bộ test

Vì bài có **nhiều đáp án đúng**, ta kiểm tra đầu ra bằng **hàm kiểm tra** `chan_doan(s, out)` gồm:

1. **Cú pháp hợp lệ** và **giải mã lại ra đúng `s`**;
2. **Không mã hóa thừa**: mọi khối `k[e]` ngắn hơn bản giải mã của nó;
3. **Độ dài ngắn nhất**: so với `do_dai_toi_uu(s)` (oracle độc lập), được kiểm chứng ngược với oracle **vét cạn sinh mọi cách mã hóa** trên mọi chuỗi ngắn.

### 12.1. Hàm kiểm tra và oracle độ dài

In [22]:
@lru_cache(maxsize=None)
def do_dai_toi_uu(t):
    """Oracle 2: độ dài mã hóa ngắn nhất, thử MỌI ước làm chu kỳ và MỌI điểm cắt (không giả định chu kỳ nhỏ nhất)."""
    n = len(t)
    best = n
    for d in range(1, n):
        if n % d == 0 and t[:d] * (n // d) == t:
            best = min(best, len(str(n // d)) + 2 + do_dai_toi_uu(t[:d]))
    for i in range(1, n):
        best = min(best, do_dai_toi_uu(t[:i]) + do_dai_toi_uu(t[i:]))
    return best

def chan_doan(s, out):
    """Trả '' nếu out là đáp án hợp lệ cho s; ngược lại trả lý do."""
    pt = phan_tich(out)
    if pt is None:
        return "sai cú pháp"
    giai, vi_pham = pt
    if giai != s:
        return f"giải mã ra {giai!r} khác chuỗi gốc"
    if vi_pham:
        return "có khối mã hóa không làm ngắn đi (vi phạm 'không ngắn hơn thì đừng mã hóa')"
    toi_uu = do_dai_toi_uu(s)
    if len(out) != toi_uu:
        return f"độ dài {len(out)} khác độ dài ngắn nhất {toi_uu}"
    return ""

def hop_le(s, out):
    return chan_doan(s, out) == ""

print(chan_doan("aaaaa", "5[a]") or "OK", "|", chan_doan("aaaaa", "aaaaa"), "|", chan_doan("aaaaa", "4[a]a"))
print(chan_doan("aaaa", "4[a]"))
print(chan_doan("abbbabbbcabbbabbbc", "2[abbbabbbc]"))
print(chan_doan("aaaaaaaaaa", "10[a]") or "OK", chan_doan("aaaaaaaaaa", "a9[a]") or "OK", chan_doan("aaaaaaaaaa", "9[a]a") or "OK")

OK | độ dài 5 khác độ dài ngắn nhất 4 | có khối mã hóa không làm ngắn đi (vi phạm 'không ngắn hơn thì đừng mã hóa')
có khối mã hóa không làm ngắn đi (vi phạm 'không ngắn hơn thì đừng mã hóa')
độ dài 12 khác độ dài ngắn nhất 11
OK OK OK


### 12.2. Kiểm chứng oracle độ dài bằng vét cạn (độc lập với DP)

`do_dai_toi_uu` dùng đệ quy; để **tin được** nó, ta so với oracle **sinh mọi cách mã hóa** (Cách 1) trên **mọi** chuỗi `ab` độ dài `1..9` và mọi chuỗi `abc` độ dài `1..6`.

In [23]:
def moi_chuoi(bang, toi_da):
    for n in range(1, toi_da + 1):
        for tup in itertools.product(bang, repeat=n):
            yield "".join(tup)

dem = 0
for s in itertools.chain(moi_chuoi("ab", 9), moi_chuoi("abc", 6)):
    vet_can = min(len(e) for e in sinh_tat_ca(s))
    assert vet_can == do_dai_toi_uu(s), (s, vet_can, do_dai_toi_uu(s))
    dem += 1
print("oracle độ dài khớp vét cạn trên", dem, "chuỗi")

oracle độ dài khớp vét cạn trên 2114 chuỗi


### 12.3. Ví dụ của đề, edge case và ví dụ kinh điển

In [24]:
sol = Solution()
enc = sol.encode

for s, dai in examples:
    assert hop_le(s, enc(s)) and len(enc(s)) == dai

# các chuỗi ngắn (<= 4) phải giữ nguyên
for s in ["a", "ab", "aa", "aaa", "aaaa", "abab", "abca", "zzzz", "abcd"]:
    assert enc(s) == s, (s, enc(s))

# kinh điển
for s, dai in [
    ("aaaaa", 4), ("aaaaaa", 4), ("aaaaaaaaaa", 5), ("abababab", 5), ("aabcaabcd", 8),
    ("abbbabbbcabbbabbbc", 11), ("aaaaaaaaaabcbcbcbcbc", 10), ("abcabcabcabc", 6),
]:
    out = enc(s)
    assert hop_le(s, out) and len(out) == dai, (s, out)
    print(f"{s!r:24} -> {out!r}")

# độ dài tối đa của đề
for s, mong in [("a" * 150, "150[a]"), ("ab" * 75, "75[ab]")]:
    assert enc(s) == mong, (enc(s), mong)
    assert hop_le(s, enc(s))
print("Ví dụ, edge case, kinh điển OK")

'aaaaa'                  -> '5[a]'
'aaaaaa'                 -> '6[a]'
'aaaaaaaaaa'             -> '10[a]'
'abababab'               -> '4[ab]'
'aabcaabcd'              -> '2[aabc]d'
'abbbabbbcabbbabbbc'     -> '2[2[abbb]c]'
'aaaaaaaaaabcbcbcbcbc'   -> 'a9[a]5[bc]'
'abcabcabcabc'           -> '4[abc]'


Ví dụ, edge case, kinh điển OK


### 12.4. Đối chiếu các cách trên mọi chuỗi ngắn (vét cạn)

Với **mọi** chuỗi `ab` độ dài `1..10` và mọi chuỗi `abc` độ dài `1..7`: Cách 1 (vét cạn sinh), Cách 2, Cách 3, Cách 4 và hàm `Solution` phải cho đáp án **hợp lệ** (kiểm bằng `chan_doan`, tức đúng độ dài ngắn nhất theo oracle).

In [25]:
cac_cach = {
    "Cách 2 (memo)": encode_memo, "Cách 3 (DP)": encode_dp, "Cách 4 (KMP)": encode_kmp,
    "Solution": Solution().encode, "SolutionMemo": SolutionMemo().encode,
    "SolutionDP": SolutionDP().encode, "SolutionKMP": SolutionKMP().encode,
}
dem = 0
for s in itertools.chain(moi_chuoi("ab", 10), moi_chuoi("abc", 7)):
    if len(s) <= 9:                                           # Cách 1 chỉ chạy trên chuỗi ngắn
        assert hop_le(s, encode_vet_can(s)), s
    for ten, fn in cac_cach.items():
        out = fn(s)
        ly_do = chan_doan(s, out)
        assert not ly_do, (ten, s, out, ly_do)
    dem += 1
print("Mọi cách cho đáp án hợp lệ trên", dem, "chuỗi (ab ≤ 10, abc ≤ 7)")

Mọi cách cho đáp án hợp lệ trên 5325 chuỗi (ab ≤ 10, abc ≤ 7)


### 12.5. Chuỗi có cấu trúc lặp lồng nhau (ngẫu nhiên có seed) và chuỗi dài tới 150

Sinh các chuỗi bằng một "biểu thức lặp" ngẫu nhiên (lặp lồng nhau, nối ghép), rồi kiểm tra mọi cách: đây là các chuỗi mà mã hóa ngắn nhất nhiều tầng và có nhiều đáp án đúng.

In [26]:
random.seed(471)

def sinh_cau_truc(do_sau):
    if do_sau == 0 or random.random() < 0.25:
        return "".join(random.choice("ab") for _ in range(random.randint(1, 3)))
    phan = []
    for _ in range(random.randint(1, 3)):
        con = sinh_cau_truc(do_sau - 1)
        phan.append(con * random.randint(2, 4) if random.random() < 0.6 else con)
    return "".join(phan)

kiem_tra_xong = 0
so_co_long = 0
while kiem_tra_xong < 400:
    s = sinh_cau_truc(3)
    if not 5 <= len(s) <= 60:
        continue
    for ten, fn in cac_cach.items():
        assert hop_le(s, fn(s)), (ten, s, fn(s))
    out = Solution().encode(s)
    so_co_long += out.count("[") >= 2
    kiem_tra_xong += 1
print("400 chuỗi có cấu trúc: tất cả hợp lệ; số đáp án có lồng/nhiều khối:", so_co_long)
assert so_co_long > 50

400 chuỗi có cấu trúc: tất cả hợp lệ; số đáp án có lồng/nhiều khối: 221


In [27]:
# Chuỗi dài tới 150: các dạng xấu nhất
random.seed(1471)
dai = {
    "a*150": "a" * 150,
    "ab*75": "ab" * 75,
    "abc*50": "abc" * 50,
    "aaab lặp": ("aaab" * 10 + "c") * 3 + "d" * 5,
    "ngẫu nhiên ab": "".join(random.choice("ab") for _ in range(150)),
    "ngẫu nhiên 26 chữ": "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(150)),
    "cấu trúc lặp lồng": (sinh_cau_truc(3) * 20)[:150],
}
for ten, s in dai.items():
    assert len(s) <= 150
    t0 = time.perf_counter()
    out = Solution().encode(s)
    t1 = time.perf_counter()
    assert hop_le(s, out), ten
    assert len(encode_kmp(s)) == len(out) == len(encode_memo(s))
    print(f"{ten:<20} n={len(s):>3}: kết quả dài {len(out):>3}  {1000 * (t1 - t0):7.1f} ms  {out[:40]!r}")

a*150                n=150: kết quả dài   6     52.7 ms  '150[a]'


ab*75                n=150: kết quả dài   6     52.9 ms  '75[ab]'


abc*50               n=150: kết quả dài   7     52.9 ms  '50[abc]'


aaab lặp             n=128: kết quả dài  16     35.9 ms  '3[10[aaab]c]5[d]'


ngẫu nhiên ab        n=150: kết quả dài 130     51.9 ms  'a2[aaaba]babbbbaaaa2[bbaba]aaabbbbababbb'


ngẫu nhiên 26 chữ    n=150: kết quả dài 150     52.3 ms  'efxcmhnbupkqknzrbljjkeqmnmgnkdgqqovzdyhw'
cấu trúc lặp lồng    n= 40: kết quả dài   6      1.2 ms  '20[ab]'


## 13. Các lỗi người mới hay gặp

Với bài nhiều đáp án đúng, mỗi lỗi dưới đây được **bắt bằng hàm kiểm tra** `chan_doan`.

### Lỗi 1 — Mã hóa cả khi không ngắn đi thật sự (`aabaab` → `2[aab]`)

Điều kiện `k < L` (có chu kỳ) thôi chưa đủ: phải so **độ dài sau mã hóa** với độ dài gốc.

In [28]:
def sai_khong_nghiem_ngat(s):
    n = len(s)
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            f[i][j] = t
            if len(t) >= 5:
                p = (t + t).find(t, 1)
                if p < len(t):
                    f[i][j] = f"{len(t) // p}[{f[i][i + p - 1]}]"      # SAI: không so độ dài
                for m in range(i, j):
                    if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                        f[i][j] = f[i][m] + f[m + 1][j]
    return f[0][n - 1]

s = "aabaab"
print("sai :", sai_khong_nghiem_ngat(s), "->", chan_doan(s, sai_khong_nghiem_ngat(s)))
print("đúng:", enc(s), "->", chan_doan(s, enc(s)) or "OK")
assert sai_khong_nghiem_ngat(s) == "2[aab]" and "không làm ngắn" in chan_doan(s, "2[aab]")
assert enc(s) == "aabaab"

sai : 2[aab] -> có khối mã hóa không làm ngắn đi (vi phạm 'không ngắn hơn thì đừng mã hóa')
đúng: aabaab -> OK


**Vì sao sai:** `"2[aab]"` dài `6` bằng `"aabaab"` nên không "ngắn đi" mà đề dặn **đừng mã hóa**. **Cách đúng:** chỉ nhận ứng viên khi `len(cand) < len(hiện tại)`.

### Lỗi 2 — Bọc chuỗi **thô** thay vì mã hóa tối ưu bên trong

`"abbbabbbcabbbabbbc"` bọc thô thành `"2[abbbabbbc]"` (12), nhưng tối ưu là `"2[2[abbb]c]"` (11): phần trong ngoặc cũng phải được mã hóa.

In [29]:
def sai_noi_dung_tho(s):
    t = s
    if len(t) >= 5:
        p = (t + t).find(t, 1)
        if p < len(t):
            cand = f"{len(t) // p}[{t[:p]}]"                           # SAI: t[:p] chưa mã hóa
            if len(cand) < len(t):
                return cand
    return t

s = "abbbabbbcabbbabbbc"
print("sai :", sai_noi_dung_tho(s), "->", chan_doan(s, sai_noi_dung_tho(s)))
print("đúng:", enc(s), "->", chan_doan(s, enc(s)) or "OK")
assert chan_doan(s, sai_noi_dung_tho(s)).startswith("độ dài 12") and len(enc(s)) == 11

sai : 2[abbbabbbc] -> độ dài 12 khác độ dài ngắn nhất 11
đúng: 2[2[abbb]c] -> OK


**Vì sao sai:** phần được lặp cũng có thể ngắn đi nếu mã hóa tiếp. **Cách đúng:** dùng `f[i][i+p-1]` (mã hóa **tối ưu** của phần lặp, đã tính sẵn) bên trong ngoặc.

### Lỗi 3 — Quên bước cắt đôi

Không thử cắt đôi thì chuỗi như `"aaaaaaaaaabcbcbcbcbc"` (= `10[a]` rồi `5[bc]`) không được mã hóa vì **toàn chuỗi không lặp**.

In [30]:
def sai_khong_cat(s):
    n = len(s)
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            f[i][j] = t
            if len(t) >= 5:
                p = (t + t).find(t, 1)
                if p < len(t):
                    cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
                    if len(cand) < len(t):
                        f[i][j] = cand
                # SAI: thiếu vòng cắt đôi
    return f[0][n - 1]

s = "aaaaaaaaaabcbcbcbcbc"
print("sai :", sai_khong_cat(s), "->", chan_doan(s, sai_khong_cat(s)))
print("đúng:", enc(s), "->", chan_doan(s, enc(s)) or "OK")
assert chan_doan(s, sai_khong_cat(s)).startswith("độ dài 20") and len(enc(s)) == 10

sai : aaaaaaaaaabcbcbcbcbc -> độ dài 20 khác độ dài ngắn nhất 10
đúng: a9[a]5[bc] -> OK


**Vì sao sai:** mã hóa tối ưu có thể là **ghép** nhiều khối. **Cách đúng:** thử mọi điểm cắt `m` với `f[i][m] + f[m+1][j]`.

### Lỗi 4 — Chọn "tốt nhất" bằng `min` trên chuỗi (so thứ tự từ điển) thay vì theo độ dài

`min(["abcde", "5[a]"])` trả `"5[a]"` chỉ vì ký tự `'5'` nhỏ hơn `'a'` trong bảng mã, **không phải** vì nó ngắn hơn. Có lúc cách đó chọn mã hóa **dài hơn**.

In [31]:
print(min(["abcde", "5[a]"]), "| nhưng:", min(["abc", "2[a]b"]), "<- chọn theo từ điển: '2[a]b' dài hơn 'abc'?")

def sai_min_tu_dien(s):
    memo = {}
    def e(t):
        if t in memo:
            return memo[t]
        ung_vien = [t]
        if len(t) >= 5:
            p = (t + t).find(t, 1)
            if p < len(t):
                ung_vien.append(f"{len(t) // p}[{e(t[:p])}]")
            for i in range(1, len(t)):
                ung_vien.append(e(t[:i]) + e(t[i:]))
        memo[t] = min(ung_vien)                                        # SAI: min theo từ điển
        return memo[t]
    return e(s)

phan_vi = None
for s in moi_chuoi("ab", 8):
    if not hop_le(s, sai_min_tu_dien(s)):
        phan_vi = s
        break
print("chuỗi ngắn nhất làm lời giải sai vỡ:", repr(phan_vi), "->", repr(sai_min_tu_dien(phan_vi)), "|", chan_doan(phan_vi, sai_min_tu_dien(phan_vi)))
assert phan_vi is not None and hop_le(phan_vi, enc(phan_vi))

5[a] | nhưng: 2[a]b <- chọn theo từ điển: '2[a]b' dài hơn 'abc'?
chuỗi ngắn nhất làm lời giải sai vỡ: 'aaaaaa' -> '5[a]a' | độ dài 5 khác độ dài ngắn nhất 4


**Vì sao sai:** thứ tự từ điển không liên quan độ dài (chữ số `<` chữ cái, nên ứng viên bắt đầu bằng số luôn "thắng"). **Cách đúng:** so sánh bằng `len(...)`.

### Lỗi 5 (kỹ thuật kiểm thử) — So chuỗi kết quả với một đáp án cứng

Với bài nhiều đáp án đúng, `assert enc("aaaaaaaaaa") == "10[a]"` có thể sai oan khi bạn đổi cách xử lý hòa (ví dụ ra `"a9[a]"`). Hãy kiểm bằng `hop_le`.

In [32]:
for out in ["10[a]", "a9[a]", "9[a]a"]:
    assert hop_le("aaaaaaaaaa", out)
print("cả 3 đáp án đều hợp lệ; ta kiểm tra bằng hàm hop_le thay vì so chuỗi cứng")
print("Solution trả:", enc("aaaaaaaaaa"))

cả 3 đáp án đều hợp lệ; ta kiểm tra bằng hàm hop_le thay vì so chuỗi cứng
Solution trả: 10[a]


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "chia một chuỗi/đoạn thành các phần và chi phí mỗi phần độc lập" + `n` nhỏ (`<= 150`) ⇒ **DP theo đoạn `[i, j]`** với hai kiểu chuyển: cắt đôi, hoặc xử lý toàn đoạn.
- **Chu kỳ của chuỗi:** `(t + t).find(t, 1) < len(t)` (bài 459 Repeated Substring Pattern), hoặc `L - pi[L-1]` từ hàm tiền tố KMP: nhớ cả hai.
- **Bài có nhiều đáp án đúng:** tự viết **hàm kiểm tra** (giải mã, ràng buộc, tối ưu) rồi so với **oracle độc lập** trên chuỗi ngắn.
- **So sánh theo độ dài trước khi ghép chuỗi** để tránh tạo chuỗi trung gian vô ích: `len(a) + len(b) < len(best)`.
- **Đừng quên điều kiện "ngắn đi thật sự"** khi đề dặn "nếu không ngắn hơn thì đừng làm".
- **Ngưỡng nhỏ:** biết chuỗi `< 5` không đáng mã hóa giúp bỏ qua một nửa việc cho đoạn ngắn.

## 15. Biến thể

**Biến thể A — Repeated Substring Pattern (LeetCode 459):** chuỗi có phải là một chuỗi con lặp `>= 2` lần không? Dùng đúng mẹo `(s + s).find(s, 1) < len(s)`.

**Biến thể B — Decode String (LeetCode 394):** giải mã `"3[a2[c]]"` → `"accaccacc"`; chính là bộ giải mã bằng ngăn xếp ở mục 3.2 và `phan_tich` ở mục 5.

**Biến thể C — chỉ tính độ dài ngắn nhất:** không cần dựng chuỗi, DP trên số nguyên: `f[i][j] = min(L, len(str(k)) + 2 + f[i][i+p-1], f[i][m] + f[m+1][j])`. Tiết kiệm bộ nhớ.

In [33]:
def lap_lai(s):
    """Biến thể A (459): s có phải là một chuỗi con lặp lại >= 2 lần?"""
    return (s + s).find(s, 1) < len(s)

def do_dai_dp(s):
    """Biến thể C: chỉ tính độ dài mã hóa ngắn nhất bằng DP trên số nguyên."""
    n = len(s)
    f = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            L = j - i + 1
            best = L
            if L >= 5:
                t = s[i:j + 1]
                p = (t + t).find(t, 1)
                if p < L:
                    best = min(best, len(str(L // p)) + 2 + f[i][i + p - 1])
                for m in range(i, j):
                    best = min(best, f[i][m] + f[m + 1][j])
            f[i][j] = best
    return f[0][n - 1]

print(lap_lai("abab"), lap_lai("aba"), lap_lai("abcabcabc"), lap_lai("a"))
assert lap_lai("abab") and not lap_lai("aba") and lap_lai("abcabcabc") and not lap_lai("a")
print(giai_ma_don_gian("3[a2[c]]"))
assert giai_ma_don_gian("3[a2[c]]") == "accaccacc"
random.seed(12)
for _ in range(300):
    s = sinh_cau_truc(3)[:60]
    assert do_dai_dp(s) == len(enc(s)) == do_dai_toi_uu(s)
print("do_dai_dp khớp độ dài của Solution và oracle trên 300 chuỗi")

True False True False
accaccacc


do_dai_dp khớp độ dài của Solution và oracle trên 300 chuỗi


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh mọi cách mã hóa | Dựa thẳng định nghĩa, lọc và chọn ngắn nhất | hàm mũ | hàm mũ | **Oracle** cho chuỗi rất ngắn |
| 2. Đệ quy nhớ theo chuỗi con (mọi ước) | Top-down, khóa là chuỗi con, thử mọi ước làm chu kỳ | `~O(n³)` | `O(n²)` | Dễ viết; không giả định chu kỳ nhỏ nhất |
| 3. DP đoạn + `(t+t).find` | Bottom-up `f[i][j]`, chu kỳ nhỏ nhất bằng `find` | `O(n³)` | `O(n²)` | **Cách nộp bài** |
| 4. DP đoạn + KMP | Giống Cách 3, chu kỳ bằng `L - pi[L-1]` | `O(n³)` (chu kỳ `O(n²)`) | `O(n²)` | Khi muốn bỏ chi phí `find`; học KMP |

In [34]:
random.seed(7)
mau = {
    "a*150": "a" * 150,
    "ngẫu nhiên ab": "".join(random.choice("ab") for _ in range(150)),
    "cấu trúc": (sinh_cau_truc(3) * 40)[:150],
}
for ten_mau, s in mau.items():
    print(f"--- {ten_mau} (n={len(s)})")
    for ten, fn in [("Cách 2", encode_memo), ("Cách 3", encode_dp), ("Cách 4", encode_kmp)]:
        t0 = time.perf_counter()
        out = fn(s)
        print(f"   {ten}: {1000 * (time.perf_counter() - t0):8.1f} ms, độ dài kết quả {len(out)}")

--- a*150 (n=150)
   Cách 2:      4.8 ms, độ dài kết quả 6
   Cách 3:     56.6 ms, độ dài kết quả 6
   Cách 4:     51.0 ms, độ dài kết quả 6
--- ngẫu nhiên ab (n=150)


   Cách 2:    232.5 ms, độ dài kết quả 115
   Cách 3:     52.8 ms, độ dài kết quả 115
   Cách 4:     49.8 ms, độ dài kết quả 115
--- cấu trúc (n=150)


   Cách 2:    134.7 ms, độ dài kết quả 37
   Cách 3:     51.3 ms, độ dài kết quả 37
   Cách 4:     49.2 ms, độ dài kết quả 37


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `giai_ma_don_gian("2[ab]c")`; (b) `len("10[a]")`; (c) `(t + t).find(t, 1)` với `t = "abcabc"`; (d) `Solution().encode("abcd")`; (e) độ dài của `Solution().encode("aaaaa")`.

**Bài 2 (tính tay).** Với `s = "abababab"` (8 ký tự): tìm chu kỳ nhỏ nhất, ứng viên bọc, và so với ứng viên lồng `2[2[ab]]`; đáp án ngắn nhất là gì và dài bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành `chu_ky_nho_nhat(t)` trả về chu kỳ nhỏ nhất `p` của `t` (hoặc `len(t)` nếu không lặp).

In [35]:
def chu_ky_nho_nhat(t):
    # TODO 1: p = (t + t).find(t, 1)
    # TODO 2: trả về p (nếu p == len(t) nghĩa là không lặp)
    return None

# KHÔNG gọi chu_ky_nho_nhat ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `encode_memo_todo(s)` theo Cách 2 (đệ quy có nhớ theo chuỗi con, thử mọi ước) **mà không nhìn lại** mục 7. Kiểm tra bằng `hop_le` trên vài chuỗi.

**Bài 5 (biến thể / mini challenge).** Viết `dem_khoi(out)` đếm số khối `k[...]` (tức số dấu `[`) trong một chuỗi mã hóa, rồi viết `encode_it_khoi(s)` trả về đáp án ngắn nhất **có ít khối nhất** (hòa độ dài thì ưu tiên ít khối). Gợi ý: sửa DP: so sánh khóa `(len, số khối)` thay vì chỉ `len`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) `"ababc"`; (b) `5`; (c) `3`; (d) chuỗi `< 5` giữ nguyên: `"abcd"`; (e) `4` (`"5[a]"`). Bài 3: chỉ cần hai dòng.
- **Mức 2 (vừa):** Bài 2: chu kỳ nhỏ nhất của `"abababab"` là `2` (vì `"ab"` lặp 4 lần): ứng viên `4[ab]` dài `5`; còn `2[2[ab]]` dài `8`; vậy chọn `4[ab]`. Bài 4: nhớ so **độ dài** trước khi ghép chuỗi.
- **Mức 3 (sâu):** Bài 5: lưu trong `f[i][j]` một chuỗi, nhưng khi so sánh dùng `(len(x), x.count("["))`. Với `"aaaaaaaaaa"` ba đáp án `10[a]`, `a9[a]`, `9[a]a` đều có độ dài `5`, nhưng `10[a]` chỉ có **1** khối còn các đáp án khác cũng 1 khối; thử `s = "abababababab"` (12 ký tự) để thấy khác biệt tiềm năng.

## 19. Đáp án tham khảo cho bài tập

In [36]:
# Bài 1
assert giai_ma_don_gian("2[ab]c") == "ababc"
assert len("10[a]") == 5
assert ("abcabc" + "abcabc").find("abcabc", 1) == 3
assert enc("abcd") == "abcd"
assert len(enc("aaaaa")) == 4
print("Bài 1 OK")

# Bài 2
t = "abababab"
p = (t + t).find(t, 1)
cand = f"{len(t) // p}[{t[:p]}]"
print("Bài 2: p =", p, "ứng viên bọc =", cand, "dài", len(cand), "| lồng 2[2[ab]] dài", len("2[2[ab]]"))
assert p == 2 and cand == "4[ab]" and len(cand) == 5
assert hop_le(t, enc(t)) and len(enc(t)) == 5
print("Bài 2 OK")

# Bài 3
def chu_ky_nho_nhat_ans(t):
    return (t + t).find(t, 1)

assert [chu_ky_nho_nhat_ans(x) for x in ["abcabc", "aaaaa", "abab", "abcab", "abcd"]] == [3, 1, 2, 5, 4]
print("Bài 3 OK")

# Bài 4
def encode_memo_todo_ans(s):
    memo = {}
    def e(t):
        if t in memo:
            return memo[t]
        n = len(t)
        best = t
        if n >= 5:
            for d in range(1, n):
                if n % d == 0 and t[:d] * (n // d) == t:
                    cand = f"{n // d}[{e(t[:d])}]"
                    if len(cand) < len(best):
                        best = cand
            for i in range(1, n):
                a, b = e(t[:i]), e(t[i:])
                if len(a) + len(b) < len(best):
                    best = a + b
        memo[t] = best
        return best
    return e(s)

random.seed(4)
for _ in range(200):
    s = sinh_cau_truc(3)[:50]
    assert hop_le(s, encode_memo_todo_ans(s))
print("Bài 4 OK")

# Bài 5
def dem_khoi(out):
    return out.count("[")

def encode_it_khoi(s):
    n = len(s)
    khoa = lambda x: (len(x), dem_khoi(x))
    f = [[""] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            t = s[i:j + 1]
            f[i][j] = t
            if len(t) >= 5:
                p = (t + t).find(t, 1)
                if p < len(t):
                    cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
                    if khoa(cand) < khoa(f[i][j]):
                        f[i][j] = cand
                for m in range(i, j):
                    cand = f[i][m] + f[m + 1][j]
                    if khoa(cand) < khoa(f[i][j]):
                        f[i][j] = cand
    return f[0][n - 1]

assert dem_khoi("2[2[ab]]") == 2 and dem_khoi("abc") == 0
random.seed(5)
for _ in range(200):
    s = sinh_cau_truc(3)[:50]
    out = encode_it_khoi(s)
    assert hop_le(s, out)                                     # vẫn ngắn nhất và hợp lệ
    assert dem_khoi(out) <= dem_khoi(enc(s))                  # không nhiều khối hơn đáp án Solution
print("Bài 5 OK: ví dụ", encode_it_khoi("aaaaaaaaaa"), encode_it_khoi("abababababab"))

Bài 1 OK
Bài 2: p = 2 ứng viên bọc = 4[ab] dài 5 | lồng 2[2[ab]] dài 8
Bài 2 OK
Bài 3 OK


Bài 4 OK


Bài 5 OK: ví dụ 10[a] 6[ab]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (DP đoạn tối ưu hóa chuỗi: bọc cả đoạn hoặc cắt đôi):**

```python
n = len(s)
f = [[""] * n for _ in range(n)]
for i in range(n - 1, -1, -1):
    for j in range(i, n):
        t = s[i:j + 1]
        f[i][j] = t
        if j - i + 1 < 5:
            continue
        p = (t + t).find(t, 1)
        if p < len(t):
            cand = f"{len(t) // p}[{f[i][i + p - 1]}]"
            if len(cand) < len(f[i][j]):
                f[i][j] = cand
        for m in range(i, j):
            if len(f[i][m]) + len(f[m + 1][j]) < len(f[i][j]):
                f[i][j] = f[i][m] + f[m + 1][j]
return f[0][n - 1]
```

**Checklist:**

- [ ] Duyệt `i` từ phải sang trái, `j` từ trái sang phải (đoạn con luôn sẵn sàng).
- [ ] Đoạn dài `< 5` giữ nguyên; chỉ nhận ứng viên khi **ngắn hơn thật sự** (`<`).
- [ ] Bọc cả đoạn bằng chu kỳ nhỏ nhất `p = (t+t).find(t, 1)` với phần trong ngoặc là `f[i][i+p-1]` (đã tối ưu), **không phải** chuỗi thô.
- [ ] Có bước **cắt đôi** `f[i][m] + f[m+1][j]` và so sánh theo `len`.
- [ ] Kiểm thử bằng hàm kiểm tra (giải mã đúng, không mã hóa thừa, độ dài bằng oracle), không so đáp án cứng.
- [ ] Thử tay `"aaa"`, `"aaaaa"`, `"aaaaaaaaaa"`, `"abbbabbbcabbbabbbc"`, `"aabcaabcd"`.
- [ ] Đúng signature `class Solution` / `encode(self, s: str) -> str`; độ phức tạp `O(n³)` thời gian, `O(n²)` bộ nhớ với `n <= 150`.